# meas_crowd with the sparse Eigen solver

Runs the pipeline on one cutout with `jointSolver="eigen"`: pipe_crowd's sparse
`LeastSquaresConjugateGradient` solver (C++), with fluxes kept >= 0.  It is the default, so
nothing else changes compared with a normal run.  Run the cells in order with the **LSST** kernel.

`compare_solvers.ipynb` runs the same cutout with the old solver too, if you want the comparison.

## 0. Set up

Set `MEAS_CROWD` to the checkout if this notebook is not in its `examples/` folder.

In [ ]:
%matplotlib inline
import os
import subprocess
import sys
import time

# where this repository is checked out (default: the folder above this notebook)
MEAS_CROWD = os.environ.get("MEAS_CROWD_DIR") or os.path.abspath("..")
# MEAS_CROWD = "/sdf/data/rubin/user/abudlong/code/build/meas_crowd_v2"
PKG = os.path.join(MEAS_CROWD, "python")
if not os.path.isdir(os.path.join(PKG, "lsst", "meas", "crowd")):
    raise FileNotFoundError(f"no meas_crowd package under {PKG}: set MEAS_CROWD above")
if PKG not in sys.path:
    sys.path.insert(0, PKG)
import lsst  # noqa: E402
import lsst.meas  # noqa: E402
for mod, sub in ((lsst, "lsst"), (lsst.meas, os.path.join("lsst", "meas"))):
    if os.path.join(PKG, sub) not in list(mod.__path__):   # lsst / lsst.meas were imported before
        mod.__path__.append(os.path.join(PKG, sub))

Build the C++ extension if it is missing (or older than the solver).  **If it builds, restart the
kernel and run from the top.**

In [ ]:
import glob

CHECK = """
import importlib.util, sys
spec = importlib.util.spec_from_file_location("_crowdLib", sys.argv[1])
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
print(hasattr(m, "solveSparseLeastSquares"))
"""

def solverBuilt():
    """True if the compiled extension has the sparse solver (checked in a fresh process)."""
    for so in glob.glob(os.path.join(PKG, "lsst", "meas", "crowd", "_crowdLib*.so")):
        out = subprocess.run([sys.executable, "-c", CHECK, so], capture_output=True, text=True)
        if out.stdout.strip() == "True":
            return True
    return False

if solverBuilt():
    print("C++ extension with the sparse solver: OK")
else:
    print("building the C++ extension ...")
    r = subprocess.run([sys.executable, "setup.py", "build_ext", "--inplace"], cwd=MEAS_CROWD,
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-3000:])
    if r.returncode == 0 and solverBuilt():
        print("\nBUILT -- now restart the kernel (Kernel > Restart) and run from the top.")
    else:
        raise RuntimeError("build failed: see the output above")

In [ ]:
import numpy as np  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
from scipy.spatial import cKDTree  # noqa: E402

import lsst.meas.crowd as crowd  # noqa: E402

print("meas_crowd from:", os.path.dirname(crowd.__file__))
print("C++ kernels:", crowd.HAVE_CXX, "| sparse Eigen solver:", crowd.HAVE_SPARSE_SOLVER)
assert crowd.HAVE_SPARSE_SOLVER, "the sparse solver is not built: run the build cell, restart"

## 1. Quick check: the solver on a synthetic crowded tile

2000 point sources of known flux in a 460 px tile; the fitted fluxes of the bright ones should
match the true fluxes to within the noise.  No data access needed; a few seconds.

In [ ]:
A, b, truth = crowd.makeCrowdedFieldProblem(nSources=2000, seed=1)
t0 = time.perf_counter()
res = crowd.solveSparse(A, b)
print(f"{A.shape[1]} sources: {time.perf_counter() - t0:.2f} s, converged {res.converged}, "
      f"{res.activeSetIterations} solves, {res.cgIterations} CG iterations, "
      f"{res.nAtBound} fluxes at zero")
bright = truth > 20 * np.median(truth)
ratio = res.x[bright] / truth[bright]
print(f"bright sources: fitted / true flux median {np.median(ratio):.4f}, "
      f"robust scatter {1.4826 * np.median(np.abs(ratio - np.median(ratio))):.4f}")
assert res.converged

## 2. Run the pipeline on a cutout

In [ ]:
VISIT, DETECTOR = 2025062100433, 144
CUTOUT = "1000,1000,300"            # x0, y0, size (detector px)
PAD = 20
PSF_SOURCE = "visit"
REPO, COLLECTION = "dp2_prep", "LSSTCam/runs/DRP/DP2"
MAX_OUTER, MAX_INNER = 3, 15
OUT = "eigen_run"
os.makedirs(OUT, exist_ok=True)

In [ ]:
from lsst.daf.butler import Butler  # noqa: E402

butler = Butler(REPO, collections=COLLECTION)
refs = butler.query_datasets("visit_image", collections="LSSTCam/runs/*", find_first=False,
                             where=f"instrument='LSSTCam' AND visit={VISIT} AND detector={DETECTOR}")
visitImage = butler.get(refs[0])

cfg = crowd.makeConfig(crowd.CROWDED_FIELD_CONFIG)
sess = crowd.prepareSession(VISIT, DETECTOR, [CUTOUT], pad=PAD, psfSource=PSF_SOURCE,
                             visitImage=visitImage)
crowd.attachGaia(sess, visitImage=visitImage, repo=REPO)      # cached in gaia_cache/

In [ ]:
crowd.solverStats(reset=True, verbose=False)
t0 = time.perf_counter()
runs = crowd.runSessionGaia(sess, cfg, outDir=OUT, maxOuter=MAX_OUTER, maxInner=MAX_INNER,
                            plotFinal=False, jointSolver="eigen")
wall = time.perf_counter() - t0
if not runs:
    raise RuntimeError("the cutout fit failed -- see the traceback printed above")
name, run = next(iter(runs.items()))
st = crowd.solverStats(verbose=False)
print(f"total {wall:.1f} s, of which joint flux fits {st['seconds']:.1f} s in {st['calls']} solves "
      f"(largest {st['maxSources']} sources, {st['fallbacks']} fallbacks to the old solver)")
print(f"{run.summary['nSources']} sources ({run.summary['nInInterior']} in the box)")
crowd.report(run)

## 3. Results

`fallbacks` above should be 0 (a fallback is a fit the sparse solver did not converge on, redone
with the old solver).

In [ ]:
crowd.plotFinalComparison(run)
plt.show()
crowd.plotResidualGray(run)
plt.show()

In [ ]:
phot = run.photometry          # saved as {OUT}/visit*_det*/<cutout>/photometry.fits
crowd.plotGaiaComparison(phot)
plt.show()
phot[:10]